# Squat Depth Checker MVP

Upload one side-view squat video, run MediaPipe pose estimation, clean the trajectory, apply simple physical consistency checks, and classify squat depth from the cleaned bottom frame.

In [ ]:
!pip install -q mediapipe opencv-python numpy matplotlib

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = 'https://github.com/AndySos/squat-depth-checker.git'
REPO_DIR = Path('/content/squat-depth-checker')
cwd = Path.cwd()

if (cwd / 'src' / 'squat_depth').exists():
    REPO_ROOT = cwd
else:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR
    os.chdir(REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
sys.path.insert(0, str(REPO_ROOT / 'src'))
print(f'Using repo at {REPO_ROOT}')

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = next(iter(uploaded.keys()))
video_path

In [ ]:
from squat_depth.pose import run_pose_landmarker
from squat_depth.temporal import clean_trajectory
from squat_depth.constraints import evaluate_constraints
from squat_depth.depth import analyze_depth
from squat_depth.visualize import save_annotated_bottom_frame, save_annotated_video

frames = run_pose_landmarker(video_path)
cleaned = clean_trajectory(frames)
constraint_report = evaluate_constraints(cleaned)
result = analyze_depth(cleaned, constraint_report)
annotated_path = save_annotated_bottom_frame(video_path, cleaned, result)
annotated_video_path = save_annotated_video(video_path, cleaned, result, constraints=constraint_report)

result, annotated_path, annotated_video_path

In [ ]:
from dataclasses import asdict
import json

summary = asdict(result)
summary.update({
    'total_frames': cleaned.frame_count,
    'constraint_flagged_frames': int(constraint_report.frame_flags.sum()),
    'jump_flagged_frames': int(cleaned.jump_flags.any(axis=1).sum()),
    'low_confidence_frames': int(cleaned.low_confidence.any(axis=1).sum()),
})
print(json.dumps(summary, indent=2))

In [ ]:
from IPython.display import Image, Video, display

display(Image(filename=str(annotated_path)))
display(Video(str(annotated_video_path), embed=True))

In [ ]:
import matplotlib.pyplot as plt
from squat_depth.depth import SIDES

joints = SIDES[result.side]
t = cleaned.timestamps_ms / 1000
plt.figure(figsize=(10, 4))
plt.plot(t, cleaned.raw_landmarks[:, joints['hip'], 1], alpha=0.35, label='raw hip y')
plt.plot(t, cleaned.landmarks[:, joints['hip'], 1], label='cleaned hip y')
plt.plot(t, cleaned.landmarks[:, joints['knee'], 1], label='cleaned knee y')
plt.axvline(result.bottom_timestamp_ms / 1000, color='black', linestyle='--', label='bottom frame')
plt.gca().invert_yaxis()
plt.legend()
plt.title('Raw vs cleaned trajectory')
plt.xlabel('seconds')
plt.ylabel('normalized image y')
plt.show()